In [19]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import HTML

from tqdm import tqdm
import os
import zipfile

import data.hts.hts as hts


#### data.hts.emp.raw

In [20]:
Q_MENAGE_COLUMNS = [ "IDENT_MEN", "pond_menC",
    "JNBVELOAD",
    "JNBVEH", "JNBMOTO", "JNBCYCLO"
]

Q_TCM_MENAGE_COLUMNS = [
    "ident_men","NPERS","pond_menC",  "decile_rev",
    "DEP_RES","REG_res",   "STATUTCOM_UU_RES"
]

K_INDIVIDU_COLUMNS = [
    "IDENT_IND", "IDENT_MEN",
    "pond_indC", "BPERMIS", "BCARTABON","ETUDIE"
]

Q_TCM_INDIVIDU_COLUMNS = [
    "ident_ind", "ident_men", "SEXE",
]
Q_TCM_INDIVIDU_KISH_COLUMNS = [
    "AGE", "ident_ind", "ident_men",
    "CS24", "SEXE", "SITUA",
]

K_DEPLOC_COLUMNS = [
    "IDENT_IND", "MMOTIFDES", "MOTPREC",
    "TYPEJOUR", "MORIHDEP", "MDESHARR", "MDISTTOT_fin",
    "MDATE_jour","MDATE_mois", "mtp",
    "REG_ORI", "REG_DES", "nb_dep",
    "POND_JOUR"
]

data_path = "data_sources"

regions = [11]
departments = []

In [21]:
# Load IRIS registry
with zipfile.ZipFile(
    f'{data_path}/emp_2019/emp_2019_donnees_individuelles_anonymisees_novembre2024.zip') as archive: 
    with archive.open("k_individu_public_V3.csv") as f:    
        df_individu = pd.read_csv(f,
            sep = ";", encoding = "latin1", usecols = K_INDIVIDU_COLUMNS,
        )    
    with archive.open("tcm_ind_public_V3.csv") as f:
        df_tcm_individu = pd.read_csv(f,
            sep = ";", encoding = "latin1", usecols = Q_TCM_INDIVIDU_COLUMNS,
        )
    with archive.open("tcm_ind_kish_public_V3.csv") as f:
        df_tcm_individu_kish = pd.read_csv(f,
            sep = ";", encoding = "latin1", usecols = Q_TCM_INDIVIDU_KISH_COLUMNS,
        )
    with archive.open("q_menage_public_V3.csv") as f:
        df_menage = pd.read_csv(f,
            sep = ";", encoding = "latin1", usecols = Q_MENAGE_COLUMNS,
            )
        
    with archive.open("tcm_men_public_V3.csv") as f:
        df_tcm_menage = pd.read_csv(f,
            sep = ",", encoding = "latin1", usecols = Q_TCM_MENAGE_COLUMNS,
            dtype = { "DEP_RES": str })
    
    with archive.open("5. k_deploc_public_V4.csv") as f:
        df_deploc = pd.read_csv(f,
            sep = ",", encoding = "latin1", usecols = K_DEPLOC_COLUMNS,
            )


#### data.hts.emp.cleaned

In [ ]:
INCOME_CLASS_BOUNDS = [400, 800, 1000, 1200, 1500, 1800, 2000, 2500, 4000, 10000, 1e6]

PURPOSE_MAP = [
    ("1", "home"),
    ("1.4", "education"),
    ("2", "shop"),
    ("3", "other"),
    ("4", "other"),
    ("5", "leisure"),
    ("6", "other"),
    ("7", "leisure"),
    ("8", "leisure"),
    ("9", "work")
]

MODES_MAP = [
    ("1", "walk"),
    ("2", "car"), #
    ("2.1", "bike"), # bike
    ("2.2", "bike"), # bike-sharing 
    ("2.4", "car_passenger"), # motorcycle passenger
    ("2.6", "car_passenger"), # same
    ("3", "car"),
    ("3.2", "car_passenger"),
    ("4", "pt"), # taxi
    ("5", "pt"),
    ("6", "pt"),
    ("7", "pt"), # Plane
    ("8", "pt"), # Boat
#    ("9", "pt") # Other
]

def convert_time(x):
    return np.dot(np.array(x.split(":"), dtype = float), [3600.0, 60.0, 1.0])


In [ ]:
# Make copies
df_persons = pd.DataFrame(df_tcm_individu, copy = True).rename(columns={"ident_ind":"IDENT_IND", "ident_men":"IDENT_MEN"})
df_households = pd.DataFrame(df_tcm_menage, copy = True).rename(columns={ "ident_men":"IDENT_MEN"})
df_trips = pd.DataFrame(df_deploc, copy = True)

# Get weights for persons that actually have trips
df_persons = pd.merge(df_persons, df_trips[["IDENT_IND", "POND_JOUR"]].drop_duplicates("IDENT_IND"), on = "IDENT_IND", how = "left")
df_persons["is_kish"] = ~df_persons["POND_JOUR"].isna()
df_persons["trip_weight"] = df_persons["POND_JOUR"].fillna(0.0)

# Merge in additional information from EMP
df_households = pd.merge(df_households, df_menage[[
    "IDENT_MEN", "JNBVELOAD",
"JNBVEH", "JNBMOTO", "JNBCYCLO"
]], on = "IDENT_MEN", how = "left")

df_persons = pd.merge(df_persons, df_tcm_individu_kish[["AGE", "ident_ind","CS24", "SITUA",
]].rename(columns={"ident_ind":"IDENT_IND"}), on = "IDENT_IND", how = "left")

df_persons = pd.merge(df_persons, df_individu[[
    "IDENT_IND", "BPERMIS", "BCARTABON","ETUDIE","pond_indC"
]], on = "IDENT_IND", how = "left")

# Transform original IDs to integer (they are hierarchichal)
df_persons["emp_person_id"] = df_persons["IDENT_IND"].astype(int)
df_persons["emp_household_id"] = df_persons["IDENT_MEN"].astype(int)
df_households["emp_household_id"] = df_households["IDENT_MEN"].astype(int)
df_trips["emp_person_id"] = df_trips["IDENT_IND"].astype(int)

# Construct new IDs for households, persons and trips (which are unique globally)
df_households["household_id"] = np.arange(len(df_households))

df_persons = pd.merge(
    df_persons, df_households[["emp_household_id", "household_id", "DEP_RES"]],
    on = "emp_household_id"
)
df_persons["person_id"] = np.arange(len(df_persons))

df_trips = pd.merge(
    df_trips, df_persons[["emp_person_id", "person_id", "household_id"]],
    on = ["emp_person_id"]
)
df_trips["trip_id"] = np.arange(len(df_trips))

# Weight
df_persons["person_weight"] = df_persons["pond_indC"].astype(float)
df_households["household_weight"] = df_households["pond_menC"].astype(float)

# Clean age
df_persons.loc[:, "age"] = df_persons["AGE"].fillna(0)

# Clean sex
df_persons.loc[df_persons["SEXE"] == 1, "sex"] = "male"
df_persons.loc[df_persons["SEXE"] == 2, "sex"] = "female"
df_persons["sex"] = df_persons["sex"].astype("category")

# Household size
df_households["household_size"] = df_households["NPERS"]

# Clean departement
df_households["departement_id"] = df_households["DEP_RES"].fillna("undefined").astype("category")
df_persons["departement_id"] = df_persons["DEP_RES"].fillna("undefined").astype("category")

df_trips["origin_departement_id"] = '00'
df_trips["destination_departement_id"] = '00'

# Clean urban type
df_households["urban_type"] = df_households["STATUTCOM_UU_RES"].replace({
    "B": "suburb",
    "C": "central_city",
    "I": "isolated_city",
    "H": "none"
})

assert np.all(~df_households["urban_type"].isna())
df_households["urban_type"] = df_households["urban_type"].astype("category")

# Clean employment
df_persons["employed"] = df_persons["SITUA"].isin([1, 2])

# Studies
# Many < 14 year old have NaN
df_persons["studies"] = df_persons["ETUDIE"].fillna(1) == 1
df_persons.loc[df_persons["age"] < 5, "studies"] = False

# Number of vehicles
df_households["number_of_vehicles"] = 0
df_households["number_of_vehicles"] += df_households["JNBVEH"].fillna(0)
df_households["number_of_vehicles"] += df_households["JNBMOTO"].fillna(0)
df_households["number_of_vehicles"] += df_households["JNBCYCLO"].fillna(0)
df_households["number_of_vehicles"] = df_households["number_of_vehicles"].astype(int)

df_households["number_of_bikes"] = df_households["JNBVELOAD"].fillna(0).astype(int)

# License
df_persons["has_license"] = df_persons["BPERMIS"] == 1

# Has subscription
df_persons["has_pt_subscription"] = df_persons["BCARTABON"] == 1

# Household income
df_households["income_class"] = df_households["decile_rev"] -1
df_households["income_class"] = df_households["income_class"].astype(int)

# Trip purpose
df_trips["following_purpose"] = "other"
df_trips["preceding_purpose"] = "other"

for prefix, activity_type in PURPOSE_MAP:
    df_trips.loc[
        df_trips["MMOTIFDES"].astype(str).str.startswith(prefix), "following_purpose"
    ] = activity_type

    df_trips.loc[
        df_trips["MOTPREC"].astype(str).str.startswith(prefix), "preceding_purpose"
    ] = activity_type

df_trips["following_purpose"] = df_trips["following_purpose"].astype("category")
df_trips["preceding_purpose"] = df_trips["preceding_purpose"].astype("category")

# Trip mode
df_trips["mode"] = "pt"

for prefix, mode in MODES_MAP:
    df_trips.loc[
        df_trips["mtp"].astype(str).str.startswith(prefix), "mode"
    ] = mode

df_trips["mode"] = df_trips["mode"].astype("category")

# Further trip attributes
df_trips["routed_distance"] = df_trips["MDISTTOT_fin"] * 1000.0
df_trips["routed_distance"] = df_trips["routed_distance"].fillna(0.0) # This should be just one within Île-de-France

# Only leave weekday trips
f = df_trips["TYPEJOUR"] == 1
print("Removing %d trips on weekends" % np.count_nonzero(~f))
df_trips = df_trips[f]

# Only leave one day per person
initial_count = len(df_trips)

df_first_day = df_trips[["person_id","MDATE_jour","MDATE_mois"]].sort_values(
    by = ["person_id", "MDATE_jour","MDATE_mois"]
).drop_duplicates("person_id")
df_trips = pd.merge(df_trips, df_first_day, how = "inner", on = ["person_id", "MDATE_jour","MDATE_mois"])

final_count = len(df_trips)
print("Removed %d trips for non-primary days" % (initial_count - final_count))

# Trip flags
df_trips = hts.compute_first_last(df_trips)

# Trip times
df_trips["departure_time"] = df_trips["MORIHDEP"].apply(convert_time).astype(float)
df_trips["arrival_time"] = df_trips["MDESHARR"].apply(convert_time).astype(float)
df_trips = hts.fix_trip_times(df_trips)

# Durations
df_trips["trip_duration"] = df_trips["arrival_time"] - df_trips["departure_time"]
hts.compute_activity_duration(df_trips)

# Add weight to trips
df_trips["trip_weight"] = df_trips["POND_JOUR"]

# Chain length
df_persons = pd.merge(
    df_persons, df_trips[["person_id", "nb_dep"]].drop_duplicates("person_id").rename(columns = { "nb_dep": "number_of_trips" }),
    on = "person_id", how = "left"
)
df_persons["number_of_trips"] = df_persons["number_of_trips"].fillna(-1).astype(int)
df_persons.loc[(df_persons["number_of_trips"] == -1) & df_persons["is_kish"], "number_of_trips"] = 0

# Passenger attribute
df_persons["is_passenger"] = df_persons["person_id"].isin(
    df_trips[df_trips["mode"] == "car_passenger"]["person_id"].unique()
)

# Drop person without right household size 
df_persons = df_persons.drop(df_persons[(df_persons["number_of_trips"] == -1) & (df_persons['household_id'].isin([1647,6182,12630]))].index)

# Calculate consumption units
hts.check_household_size(df_households, df_persons)
df_households = pd.merge(df_households, hts.calculate_consumption_units(df_persons), on = "household_id")

# Socioprofessional class
df_persons["socioprofessional_class"] = df_persons["CS24"].fillna(80).astype(int) // 10

# Fix activity types (because of 1 inconsistent emp data)
hts.fix_activity_types(df_trips)


Removed 0 trips for non-primary days
Found 98 occurences with negative duration
  of which 6 can swap departure and arrival time without conflicts with previous or following trip
  of which 1 are unlikely to cover midnight, so we swap arrival and departure time although there are conflicts
  of which 91 seem to cover midnight, so we shift arrival time by 24h
Shifting trips that should start after midnight
  Shifted 331 trips in round 1
  Shifted 30 trips in round 2
  Shifted 5 trips in round 3
  Shifted 2 trips in round 4
  Shifted 1 trips in round 5
  No more occurences where current trip is after the next
Found 3 occurences where current trip ends after next trip starts
  of which we're able to shorten 2 to make it consistent
Found 8 occurences where current trip is included in next trip
Fixing 0 inconsistent activity types
Trips with inconsistent activity types: 0


#### data.hts.emp.filtered

In [24]:
## data.spatial.codes

data_path = "data_sources"

regions = [11]
departments = []
codes_path = "codes_2023/reference_IRIS_geo2023.zip"
codes_xlsx = "reference_IRIS_geo2023.xlsx"

# Load IRIS registry
with zipfile.ZipFile(
    "{}/{}".format(data_path, codes_path)) as archive:
    with archive.open(codes_xlsx) as f:
        df_codes = pd.read_excel(f,
            skiprows = 5, sheet_name = "Emboitements_IRIS"
        )[["CODE_IRIS", "DEPCOM", "DEP", "REG"]].rename(columns = {
            "CODE_IRIS": "iris_id",
            "DEPCOM": "commune_id",
            "DEP": "departement_id",
            "REG": "region_id"
        })

df_codes["iris_id"] = df_codes["iris_id"].astype("category")
df_codes["commune_id"] = df_codes["commune_id"].astype("category")
df_codes["departement_id"] = df_codes["departement_id"].astype("category")
df_codes["region_id"] = df_codes["region_id"].astype(int)

# Filter zones
requested_regions = list(map(int, regions))
requested_departments = list(map(str, departments))

if len(requested_regions) > 0:
    df_codes = df_codes[df_codes["region_id"].isin(requested_regions)]

if len(requested_departments) > 0:
    df_codes = df_codes[df_codes["departement_id"].isin(requested_departments)]

df_codes["iris_id"] = df_codes["iris_id"].cat.remove_unused_categories()
df_codes["commune_id"] = df_codes["commune_id"].cat.remove_unused_categories()
df_codes["departement_id"] = df_codes["departement_id"].cat.remove_unused_categories()



In [25]:
filter_emp = True

In [ ]:
if filter_emp : 
    # Filter for non-residents
    requested_departments = df_codes["departement_id"].unique()
    f = df_persons["departement_id"].astype(str).isin(requested_departments) # pandas bug!
    df_persons = df_persons[f]

    # Only keep trips and households that still have a person
    df_trips = df_trips[df_trips["person_id"].isin(df_persons["person_id"].unique())]
    df_households = df_households[df_households["household_id"].isin(df_persons["household_id"])]

# Finish up
df_households = df_households[hts.HOUSEHOLD_COLUMNS + ["urban_type", "income_class"]]
df_persons = df_persons[hts.PERSON_COLUMNS]
df_trips = df_trips[hts.TRIP_COLUMNS + ["routed_distance"]]

hts.check(df_households, df_persons, df_trips)

Validating trip times...
  Trips with negative departure time: 0
  Trips with negative arrival time: 0
  Trips with negative duration: 0
  Trips that arrive after next departure: 0
  Trips that 'enter' following trip: 0
  Trips that 'exits' following trip 0
  Trips that 'are included in' following trip: 0
  Trips that 'cover' following trip: 0
  Trips that 'are after' following trip: 0
  Trips that have NaN times: 0
  => All trip times are consistent!
Trips with inconsistent activity types: 0


#### data.hts.emp.reweighted

In [35]:
# 1) Filter persons for which we don't have trip information
df_persons = df_persons[df_persons["number_of_trips"] >= 0].copy()

# 2) Override weights with the correct weights for the people which have trip information
df_persons["person_weight"] = df_persons["trip_weight"]

# We also add a Euclidean distance, as an approximation and for use in the downstream algorithms
df_trips["euclidean_distance"] = df_trips["routed_distance"] / 1.3
